# Domain 4: Prompt Engineering & Structured Output
## Task Statement 4.4: Validation & Retry Feedback Loops
**Certification Weight**: 18% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how to design closed-loop validation with targeted error feedback, bounded retries, and early termination on missing source data.

---

## 🏗️ Architecture: Closed-Loop Validation with Early Exit

```mermaid
flowchart TD
    Extracted["Claude Outputs Extracted Payload"] --> Validator{"Validator Check"}
    
    Validator -->|"Valid"| Success["Commit Extracted Data"]
    
    Validator -->|"Invalid (Schema / Constraint)"| ErrorType{"Is Error Fixable?"}
    
    ErrorType -->|"Missing source fact"| FastFail["Stop Retrying Immediately<br>Flag Missing / Nullable Field"]
    
    ErrorType -->|"Formatting / Math mismatch"| RetryCount{"Retry Count < 3?"}
    
    RetryCount -->|"Yes"| FeedBack["Inject Targeted Feedback<br>'Discount cannot exceed subtotal'"]
    FeedBack --> Extracted
    
    RetryCount -->|"No (Exhausted)"| Fallback["Escalate to Human Review"]
```

---

## ⚠️ The "Unfixable Error" Exam Trap

### Trap:
A question presents a document missing a Customer VAT Number. The schema marks `vat_number: str` as required. Claude fails validation. The system blindly retries 3 times.

### Correct Answer:
If the source document **genuinely lacks the data**, retrying will either:
1. Cause repeated failures (wasted tokens/latency), or
2. Force the model to hallucinate a fabricated number.

**Solution**: Stop immediately on missing source data and mark as unresolvable / escalate to human review.

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 4.4 Validation Retry & Feedback.")

### 🔄 Step 2: Resilient Extractor Loop Implementation
We implement the validation engine distinguishing recoverable retries from unrecoverable fast-fails.

In [ ]:
# Step 2: Validation Loop Logic
@dataclass
class ValidationReport:
    is_valid: bool
    is_recoverable: bool
    errors: List[str]


class ExtractionValidator:
    @staticmethod
    def validate(payload: Dict[str, Any]) -> ValidationReport:
        errors = []
        # Missing essential mandatory field -> UNRECOVERABLE
        if not payload.get("tax_id"):
            return ValidationReport(
                is_valid=False,
                is_recoverable=False,
                errors=["FATAL: Mandatory Tax ID is missing entirely from source document."],
            )

        # Arithmetic check -> RECOVERABLE
        subtotal = payload.get("subtotal", 0.0)
        total = payload.get("total", 0.0)
        if subtotal > total:
            errors.append(f"Arithmetic discrepancy: subtotal ${subtotal} cannot exceed total ${total}.")

        return ValidationReport(
            is_valid=len(errors) == 0,
            is_recoverable=True,
            errors=errors,
        )


class ResilientExtractorLoop:
    def __init__(self, max_retries: int = 3) -> None:
        self.max_retries = max_retries

    def run_validation_cycle(self, document_text: str, model_turns: List[Dict[str, Any]]) -> Dict[str, Any]:
        retries_used = 0
        for i, turn_payload in enumerate(model_turns):
            report = ExtractionValidator.validate(turn_payload)
            if report.is_valid:
                return {"status": "SUCCESS", "retries_used": retries_used, "data": turn_payload}
            
            # Fast fail if unrecoverable!
            if not report.is_recoverable:
                return {
                    "status": "ABORTED_UNRECOVERABLE",
                    "retries_used": retries_used,
                    "reason": report.errors[0],
                }

            retries_used += 1
            if retries_used >= self.max_retries:
                return {"status": "EXHAUSTED_RETRIES", "retries_used": retries_used, "errors": report.errors}

        return {"status": "NO_TURNS_PROVIDED"}

print(" ResilientExtractorLoop initialized.")

### 🧪 Step 3: Interactive Simulation
1. Recoverable retry: Turn 1 has math error, Turn 2 fixes it -> Success in 1 retry.
2. Unrecoverable missing tax ID -> Fast-fails immediately with 0 retries.

In [ ]:
# Step 3: Simulation
loop = ResilientExtractorLoop(max_retries=3)

# 1. Recoverable scenario
turns_recoverable = [
    {"subtotal": 100.0, "total": 80.0, "tax_id": "TX-99"},   # Math error
    {"subtotal": 100.0, "total": 110.0, "tax_id": "TX-99"},  # Corrected!
]
res_rec = loop.run_validation_cycle("Doc with tax TX-99", turns_recoverable)
print("Recoverable Scenario Result:", res_rec)

# 2. Unrecoverable scenario (Tax ID missing entirely)
turns_unrecoverable = [
    {"subtotal": 50.0, "total": 55.0, "tax_id": None}
]
res_unrec = loop.run_validation_cycle("Lunch receipt without tax info", turns_unrecoverable)
print("\nUnrecoverable Scenario Result:", res_unrec)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Recoverable errors succeed with `retries_used == 1`.
2. Unrecoverable errors fast-fail with `retries_used == 0` and status `ABORTED_UNRECOVERABLE`.

In [ ]:
# Step 4: Verification Checks
assert res_rec["status"] == "SUCCESS"
assert res_rec["retries_used"] == 1

assert res_unrec["status"] == "ABORTED_UNRECOVERABLE"
assert res_unrec["retries_used"] == 0
assert "FATAL" in res_unrec["reason"]

print(" All verification assertions passed! Retry and fast-fail verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Distinguish **recoverable** validation errors from **unrecoverable** document gaps.
- **Rule 2**: Fast-fail immediately on unrecoverable documents; do not burn retry budgets.
- **Rule 3**: Feed specific error strings back in the retry prompt to guide Claude's correction.